# The Structured Event Extractor

Today's drill covers how to force a LLM to reliably output structured data that an application can actually parse \
without crashing.

Imagine you are building a tool that scans raw company announcements and updates an internal calendar dashboard. You \
need to extract key events into a strict JSON format.

### Raw Text Data

In [ ]:
raw_announcement = """
    Hey team, a few quick updates on the schedule. First, the Q2 Product Roadmap Review 
    has been pushed to next Thursday, June 11th, at 2:00 PM EST. It will be hosted 
    by Sarah in the main conference room. Also, don't forget the company-wide All Hands 
    meeting is still on for this Friday, June 5th, at 10:00 AM EST on Zoom. Lastly, 
    DevOps is doing scheduled database maintenance on Saturday night (June 6th) 
    from 11:00 PM to Sunday 1:00 AM EST, so expect brief downtime. Thanks!
"""

### Expected Output Schema

In [13]:
[
  {
    "event_name": "String",
    "date": "YYYY-MM-DD or relative string if year is missing",
    "time": "String (e.g., 2:00 PM EST)",
    "location": "String (e.g., Zoom, Room Name, or Unknown)"
  }
]

[{'event_name': 'String',
  'date': 'YYYY-MM-DD or relative string if year is missing',
  'time': 'String (e.g., 2:00 PM EST)',
  'location': 'String (e.g., Zoom, Room Name, or Unknown)'}]

### Ollama Solution

In [14]:
import json
from typing import List
from pydantic import BaseModel, Field
import ollama

# Define desired output structure using Pydantic
class Event(BaseModel):
    event_name: str = Field(description="The formal name of the meeting or event.")
    date: str = Field(description="YYYY-MM-DD or relative string if year is missing.")
    time: str = Field(description="The time string, including timezone if provided.")
    location: str = Field(description="Zoom, Room Name, or 'Unknown' if not specified.")

class EventContainer(BaseModel):
    events: List[Event]

Call Ollama Chat with a JSON schema constraint (use terminal to pull llama3)

In [15]:
try:
    response = ollama.chat(
        model='llama3',
        messages=[
            {
                "role": "system",
                "content": "You are a precise data extraction assistant. "
                           "Extract all scheduled events from the text into the required structured format." 
                           "Return as JSON only."
            },
            {
                "role": "user",
                "content": raw_announcement
            }
        ],
        # Pass the raw JSON Schema dictionary straight into the format argument
        format=EventContainer.model_json_schema(),
        options={'temperature': 0}
    )

    # Extract and Parse response
    raw_content = response['message']['content']

    # Ollama's schema mode force valid JSON string inside the content field
    extracted_data = json.loads(raw_content)
    extracted_events = extracted_data["events"]

    print("Local Ollama Output: ", json.dumps(extracted_events, indent=2), "\n")

    # Rigor test (Programmatic validation using Pydantic)
    print("Running Rigor Test...")
    expected_keys = {"event_name", "date", "time", "location"}

    for idx, event in enumerate(extracted_events):
        assert isinstance(event, dict), f"Event at index {idx} is not a dictionary."

        current_keys = set(event.keys())
        assert current_keys == expected_keys, f"Index {idx} keys mismatch! Found {current_keys}, expected {expected_keys}."

        for key in expected_keys:
            assert isinstance(event[key], str), f"Index {idx}, Key '{key}' is not a string."
        
    print("Rigor Test Passed! Local Ollama model adhered perfectly to the structural blueprint.")

except Exception as e:
    print(f"Test Failed or Ollama Error: {str(e)}")


Local Ollama Output:  [
  {
    "event_name": "Q2 Product Roadmap Review",
    "date": "2023-06-11",
    "time": "14:00:00",
    "location": "Main Conference Room (hosted by Sarah)"
  },
  {
    "event_name": "Company-wide All Hands meeting",
    "date": "2023-06-05",
    "time": "10:00:00",
    "location": "Zoom"
  },
  {
    "event_name": "DevOps Database Maintenance",
    "date": "2023-06-06",
    "time": "23:00:00 - 01:00:00 (next day)",
    "location": ""
  }
] 

Running Rigor Test...
Rigor Test Passed! Local Ollama model adhered perfectly to the structural blueprint.
